# GDN Benchmark on Google Colab

This notebook runs the repository's clean-room GDN adapter with the common validation-only thresholding and point-wise evaluation utilities.

## Data and outputs

- Repository cloned or updated from: `https://github.com/Yan9564/INFORMS_FORD.git`
- Dataset root: `/content/drive/MyDrive/anomaly_benchmark_data`
- Result root: `/content/drive/MyDrive/anomaly_benchmark_results/gdn/`

## Execution modes

- `smoke`: small subset, one seed, few epochs, CPU/GPU auto-detect.
- `full`: complete configured dataset, multiple seeds, more epochs.

## Notes

SWaT is the primary temporal benchmark for this notebook. Ford is supported only when the researcher confirms the CSV rows are chronologically meaningful and should be evaluated as temporal windows. Test labels are never used for training, hyperparameter selection, or threshold selection.


In [14]:
from pathlib import Path
import os, sys, subprocess, json, platform, time, traceback, random, shutil
from datetime import datetime, timezone

IN_COLAB = 'google.colab' in sys.modules or Path('/content').exists()
REPO_URL = 'https://github.com/Yan9564/INFORMS_FORD.git'
BRANCH = 'main'
DATASET_ROOT = Path('/content/drive/MyDrive/anomaly_benchmark_data')
RESULT_ROOT = Path('/content/drive/MyDrive/anomaly_benchmark_results/gdn')
MODE = 'full'  # change to 'full' for archival runs
DATASET = 'swat'  # 'swat' primary; 'ford' only after chronological-order confirmation
print('In Colab:', IN_COLAB)
print(sys.version)
print(platform.platform())


In Colab: True
3.12.13 (main, Mar  4 2026, 09:23:07) [GCC 11.4.0]
Linux-6.6.122+-x86_64-with-glibc2.35


In [15]:
if IN_COLAB:
    from google.colab import drive
    drive.mount('/content/drive')
else:
    print('Local runtime detected; Google Drive mount skipped.')


Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [16]:
repo_dir = Path('/content/INFORMS_FORD') if IN_COLAB else Path.cwd()
if IN_COLAB:
    if repo_dir.exists():
        subprocess.run(['git', '-C', str(repo_dir), 'fetch', 'origin'], check=True)
        subprocess.run(['git', '-C', str(repo_dir), 'checkout', BRANCH], check=True)
        subprocess.run(['git', '-C', str(repo_dir), 'pull', '--ff-only'], check=True)
    else:
        subprocess.run(['git', 'clone', '--branch', BRANCH, REPO_URL, str(repo_dir)], check=True)
    os.chdir(repo_dir)
print('Repository:', Path.cwd())


Repository: /content/INFORMS_FORD


In [17]:
%pip install -q -r requirements.txt
%pip install -q -e .[gdn]


  Installing build dependencies ... done
  Checking if build backend supports build_editable ... done
  error: subprocess-exited-with-error
  
  × Getting requirements to build editable did not run successfully.
  │ exit code: 1
  ╰─> See above for output.
  
  note: This error originates from a subprocess, and is likely not a problem with pip.
  Getting requirements to build editable ... error
error: subprocess-exited-with-error

× Getting requirements to build editable did not run successfully.
│ exit code: 1
╰─> See above for output.

note: This error originates from a subprocess, and is likely not a problem with pip.


In [18]:
import numpy as np
import pandas as pd
import yaml

from benchmark.metrics.classification import evaluate_binary
from benchmark.metrics.thresholding import select_threshold
from benchmark.models.gdn import GDNDetector

try:
    import torch
    DEVICE = 'cuda' if torch.cuda.is_available() else 'cpu'
    print('Torch:', torch.__version__)
    print('CUDA available:', torch.cuda.is_available())
    if torch.cuda.is_available():
        print('GPU:', torch.cuda.get_device_name(0))
except Exception as exc:
    DEVICE = 'cpu'
    print('Torch import failed:', exc)


Torch: 2.11.0+cu128
CUDA available: True
GPU: Tesla T4


In [19]:
MODE_CONFIG = {
    'smoke': {'seeds': [42], 'epochs': 2, 'max_train_rows': 5000, 'max_test_rows': 5000, 'hidden_dim': 32, 'batch_size': 64},
    'full': {'seeds': [42, 43, 44], 'epochs': 20, 'max_train_rows': None, 'max_test_rows': None, 'hidden_dim': 64, 'batch_size': 128},
}
RUN_CFG = MODE_CONFIG[MODE]
LOOKBACK = 10
THRESHOLD_PERCENTILE = 95.0
RESULT_ROOT.mkdir(parents=True, exist_ok=True)
print(RUN_CFG)


{'seeds': [42, 43, 44], 'epochs': 20, 'max_train_rows': None, 'max_test_rows': None, 'hidden_dim': 64, 'batch_size': 128}


## Dataset loading

For SWaT, place normal training and attack/test CSV files under `anomaly_benchmark_data/swat/`. Update file names and label column below if your copy differs. For Ford, the notebook reuses the existing Ford loader and requires `windowing.representation='sequence'`.


In [20]:
from pathlib import Path
import zipfile

# Match the exact capitalisation used in Google Drive
SWAT_ZIP_ROOT = DATASET_ROOT / "SWaT"
SWAT_EXTRACT_ROOT = Path("/content/swat_extracted")

train_zip = SWAT_ZIP_ROOT / "SWaT_train.zip"
test_zip = SWAT_ZIP_ROOT / "SWaT_test.zip"

for path in [train_zip, test_zip]:
    if not path.exists():
        raise FileNotFoundError(f"Missing SWaT ZIP file: {path}")

train_extract_dir = SWAT_EXTRACT_ROOT / "train"
test_extract_dir = SWAT_EXTRACT_ROOT / "test"

train_extract_dir.mkdir(parents=True, exist_ok=True)
test_extract_dir.mkdir(parents=True, exist_ok=True)

with zipfile.ZipFile(train_zip, "r") as archive:
    archive.extractall(train_extract_dir)

with zipfile.ZipFile(test_zip, "r") as archive:
    archive.extractall(test_extract_dir)

train_candidates = list(train_extract_dir.rglob("*.csv"))
test_candidates = list(test_extract_dir.rglob("*.csv"))

print("Training CSV candidates:")
for path in train_candidates:
    print(" -", path)

print("\nTest CSV candidates:")
for path in test_candidates:
    print(" -", path)

if not train_candidates:
    raise FileNotFoundError(
        f"No CSV file was found inside {train_zip}"
    )

if not test_candidates:
    raise FileNotFoundError(
        f"No CSV file was found inside {test_zip}"
    )

# If an archive contains supporting CSVs, select the largest CSV,
# which should be the main dataset.
SWAT_TRAIN_CSV = max(
    train_candidates,
    key=lambda path: path.stat().st_size,
)

SWAT_TEST_CSV = max(
    test_candidates,
    key=lambda path: path.stat().st_size,
)

print("\nSelected training file:", SWAT_TRAIN_CSV)
print("Selected test file:", SWAT_TEST_CSV)

Training CSV candidates:
 - /content/swat_extracted/train/SWaT_train.csv

Test CSV candidates:
 - /content/swat_extracted/test/SWaT_test.csv

Selected training file: /content/swat_extracted/train/SWaT_train.csv
Selected test file: /content/swat_extracted/test/SWaT_test.csv


In [21]:
def load_swat():
    train_csv = SWAT_TRAIN_CSV
    test_csv = SWAT_TEST_CSV

    train_df = pd.read_csv(
        train_csv,
        nrows=RUN_CFG["max_train_rows"],
        low_memory=False,
    )

    test_df = pd.read_csv(
        test_csv,
        nrows=RUN_CFG["max_test_rows"],
        low_memory=False,
    )

    # Standardise column names
    train_df.columns = train_df.columns.astype(str).str.strip()
    test_df.columns = test_df.columns.astype(str).str.strip()

    print("Training shape:", train_df.shape)
    print("Test shape:", test_df.shape)
    print("Training columns:", train_df.columns.tolist())
    print("Test columns:", test_df.columns.tolist())

    # Detect the test label column
    label_candidates = [
        "Normal/Attack",
        "Normal/Attack.",
        "Label",
        "label",
        "Attack",
        "attack",
    ]

    label_column = next(
        (
            column
            for column in label_candidates
            if column in test_df.columns
        ),
        None,
    )

    if label_column is None:
        raise ValueError(
            "Could not detect the SWaT label column. "
            f"Available test columns: {test_df.columns.tolist()}"
        )

    print("Detected label column:", label_column)

    # Use columns shared between training and test data
    excluded_names = {
        label_column,
        "Timestamp",
        "Timestamp ",
        "timestamp",
        "Date",
        "Time",
    }

    candidate_features = [
        column
        for column in train_df.columns
        if column in test_df.columns
        and column not in excluded_names
        and "timestamp" not in column.lower()
    ]

    # Convert potential feature columns to numeric
    train_numeric = train_df[candidate_features].apply(
        pd.to_numeric,
        errors="coerce",
    )

    test_numeric = test_df[candidate_features].apply(
        pd.to_numeric,
        errors="coerce",
    )

    # Remove columns that contain no usable numeric values
    feature_cols = [
        column
        for column in candidate_features
        if train_numeric[column].notna().any()
        and test_numeric[column].notna().any()
    ]

    if not feature_cols:
        raise ValueError(
            "No common numeric feature columns were found."
        )

    print("Number of numeric features:", len(feature_cols))
    print("Feature columns:", feature_cols)

    train_values = (
        train_numeric[feature_cols]
        .ffill()
        .bfill()
        .fillna(0.0)
        .to_numpy(dtype=np.float32)
    )

    test_values = (
        test_numeric[feature_cols]
        .ffill()
        .bfill()
        .fillna(0.0)
        .to_numpy(dtype=np.float32)
    )

    # Convert test labels to 0=normal and 1=anomaly
    raw_labels = (
        test_df[label_column]
        .astype(str)
        .str.strip()
        .str.lower()
    )

    print("Raw test-label distribution:")
    print(raw_labels.value_counts(dropna=False))

    test_labels = raw_labels.map(
        lambda value: 0
        if value in {"normal", "0", "0.0", "false"}
        or "normal" in value
        else 1
    ).to_numpy(dtype=int)

    print(
        "Converted test-label distribution:",
        dict(zip(
            *np.unique(test_labels, return_counts=True)
        )),
    )

    # Preserve chronological order: first 80% train, final 20% validation
    split = max(
        LOOKBACK + 1,
        int(0.8 * len(train_values)),
    )

    if split >= len(train_values):
        raise ValueError(
            "Training data are too short for the requested LOOKBACK."
        )

    X_train, _ = make_windows(
        train_values[:split],
        labels=None,
        lookback=LOOKBACK,
    )

    # Retain LOOKBACK-1 preceding observations so the first validation
    # window has sufficient historical context.
    validation_start = split - LOOKBACK + 1

    validation_values = train_values[validation_start:]

    X_val, _ = make_windows(
        validation_values,
        labels=None,
        lookback=LOOKBACK,
    )

    y_val = np.zeros(len(X_val), dtype=int)

    X_test, y_test = make_windows(
        test_values,
        labels=test_labels,
        lookback=LOOKBACK,
    )

    source_info = {
        "train_csv": str(train_csv),
        "test_csv": str(test_csv),
        "label_column": label_column,
        "feature_count": len(feature_cols),
        "lookback": LOOKBACK,
        "chronological_split": True,
        "train_rows": len(train_values),
        "test_rows": len(test_values),
    }

    return (
        X_train,
        X_val,
        y_val,
        X_test,
        y_test,
        feature_cols,
        source_info,
    )

In [22]:
def make_windows(values, labels=None, lookback=10):
    X, y = [], []
    for start in range(0, len(values) - lookback + 1):
        X.append(values[start:start + lookback])
        if labels is not None:
            y.append(int(labels[start + lookback - 1]))
    if not X:
        raise ValueError('No windows created; reduce LOOKBACK or check data length.')
    return np.asarray(X, dtype=np.float32), (None if labels is None else np.asarray(y, dtype=int))


def load_ford():
    from benchmark.datasets.ford import FordDatasetConfig, FordDatasetLoader
    ford_root = DATASET_ROOT / 'ford' / 'Train'
    cfg = {
        'name': 'ford', 'accepted_dir': str(ford_root / 'accept'), 'rejected_dir': str(ford_root / 'reject'),
        'feature_columns': {'prefix': 'feature_', 'start': 0, 'end': 47}, 'label_column': 'feature_55',
        'normal_label': 0, 'anomaly_label': 1, 'validation_fraction': 0.2, 'split_by_file': True,
        'random_seed': 42, 'missing_values': {'strategy': 'median'},
        'clipping': {'enabled': True, 'lower_quantile': 0.001, 'upper_quantile': 0.999},
        'scaling': {'method': 'standard'}, 'windowing': {'lookback': LOOKBACK, 'stride': 1, 'representation': 'sequence'},
        'downsampling': {'every_n_rows': 1},
    }
    bundle = FordDatasetLoader(FordDatasetConfig.from_dict(cfg)).load()
    return bundle.X_train, bundle.X_validation, bundle.y_validation, bundle.X_test, bundle.y_test, bundle.feature_names, {'ford_root': str(ford_root)}

if DATASET == 'swat':
    X_train, X_val, y_val, X_test, y_test, feature_cols, source_info = load_swat()
elif DATASET == 'ford':
    X_train, X_val, y_val, X_test, y_test, feature_cols, source_info = load_ford()
else:
    raise ValueError(f'Unsupported DATASET: {DATASET}')
print('X_train', X_train.shape, 'X_val', X_val.shape, 'X_test', X_test.shape)
print('features', len(feature_cols), feature_cols[:10])
print('validation labels', dict(zip(*np.unique(y_val, return_counts=True))))
print('test labels', dict(zip(*np.unique(y_test, return_counts=True))))


Training shape: (99360, 53)
Test shape: (89984, 54)
Training columns: ['Timestamp', 'FIT101', 'LIT101', 'MV101', 'P101', 'P102', 'AIT201', 'AIT202', 'AIT203', 'FIT201', 'MV201', 'P201', 'P202', 'P203', 'P204', 'P205', 'P206', 'DPIT301', 'FIT301', 'LIT301', 'MV301', 'MV302', 'MV303', 'MV304', 'P301', 'P302', 'AIT401', 'AIT402', 'FIT401', 'LIT401', 'P401', 'P402', 'P403', 'P404', 'UV401', 'AIT501', 'AIT502', 'AIT503', 'AIT504', 'FIT501', 'FIT502', 'FIT503', 'FIT504', 'P501', 'P502', 'PIT501', 'PIT502', 'PIT503', 'FIT601', 'P601', 'P602', 'P603', 'Normal/Attack']
Test columns: ['Timestamp', 'FIT101', 'LIT101', 'MV101', 'P101', 'P102', 'AIT201', 'AIT202', 'AIT203', 'FIT201', 'MV201', 'P201', 'P202', 'P203', 'P204', 'P205', 'P206', 'DPIT301', 'FIT301', 'LIT301', 'MV301', 'MV302', 'MV303', 'MV304', 'P301', 'P302', 'AIT401', 'AIT402', 'FIT401', 'LIT401', 'P401', 'P402', 'P403', 'P404', 'UV401', 'AIT501', 'AIT502', 'AIT503', 'AIT504', 'FIT501', 'FIT502', 'FIT503', 'FIT504', 'P501', 'P502', 'PI

In [23]:
if DATASET == "swat":
    (
        X_train,
        X_val,
        y_val,
        X_test,
        y_test,
        feature_cols,
        source_info,
    ) = load_swat()
elif DATASET == "ford":
    (
        X_train,
        X_val,
        y_val,
        X_test,
        y_test,
        feature_cols,
        source_info,
    ) = load_ford()
else:
    raise ValueError(f"Unsupported DATASET: {DATASET}")

print("X_train:", X_train.shape)
print("X_val:", X_val.shape)
print("X_test:", X_test.shape)
print("Feature count:", len(feature_cols))
print(
    "Validation labels:",
    dict(zip(*np.unique(y_val, return_counts=True))),
)
print(
    "Test labels:",
    dict(zip(*np.unique(y_test, return_counts=True))),
)
print("Source information:", source_info)

Training shape: (99360, 53)
Test shape: (89984, 54)
Training columns: ['Timestamp', 'FIT101', 'LIT101', 'MV101', 'P101', 'P102', 'AIT201', 'AIT202', 'AIT203', 'FIT201', 'MV201', 'P201', 'P202', 'P203', 'P204', 'P205', 'P206', 'DPIT301', 'FIT301', 'LIT301', 'MV301', 'MV302', 'MV303', 'MV304', 'P301', 'P302', 'AIT401', 'AIT402', 'FIT401', 'LIT401', 'P401', 'P402', 'P403', 'P404', 'UV401', 'AIT501', 'AIT502', 'AIT503', 'AIT504', 'FIT501', 'FIT502', 'FIT503', 'FIT504', 'P501', 'P502', 'PIT501', 'PIT502', 'PIT503', 'FIT601', 'P601', 'P602', 'P603', 'Normal/Attack']
Test columns: ['Timestamp', 'FIT101', 'LIT101', 'MV101', 'P101', 'P102', 'AIT201', 'AIT202', 'AIT203', 'FIT201', 'MV201', 'P201', 'P202', 'P203', 'P204', 'P205', 'P206', 'DPIT301', 'FIT301', 'LIT301', 'MV301', 'MV302', 'MV303', 'MV304', 'P301', 'P302', 'AIT401', 'AIT402', 'FIT401', 'LIT401', 'P401', 'P402', 'P403', 'P404', 'UV401', 'AIT501', 'AIT502', 'AIT503', 'AIT504', 'FIT501', 'FIT502', 'FIT503', 'FIT504', 'P501', 'P502', 'PI

In [24]:
def run_seed(seed: int):
    # Reproducibility
    random.seed(seed)
    np.random.seed(seed)

    if "torch" in globals():
        torch.manual_seed(seed)

        if torch.cuda.is_available():
            torch.cuda.manual_seed_all(seed)

    # Output paths
    run_dir = RESULT_ROOT / DATASET / MODE / f"seed_{seed}"
    metrics_path = run_dir / "metrics.json"
    failure_path = run_dir / "failure.log"

    # Resume support
    if metrics_path.exists():
        print(f"Skipping completed seed {seed}: {metrics_path}")
        return json.loads(
            metrics_path.read_text(encoding="utf-8")
        )

    run_dir.mkdir(parents=True, exist_ok=True)
    start_time = time.perf_counter()

    try:
        # Initialise model
        model = GDNDetector(
            epochs=RUN_CFG["epochs"],
            learning_rate=0.001,
            hidden_dim=RUN_CFG["hidden_dim"],
            batch_size=RUN_CFG["batch_size"],
            top_k=max(
                1,
                min(5, X_train.shape[-1] - 1),
            ),
            device=DEVICE,
            random_state=seed,
        )

        # Train
        train_start = time.perf_counter()
        model.fit(X_train)
        train_runtime = time.perf_counter() - train_start

        # Generate validation and test anomaly scores
        score_start = time.perf_counter()

        val_scores = model.score_samples(X_val)
        test_scores = model.score_samples(X_test)

        score_runtime = time.perf_counter() - score_start

        # Select threshold using validation scores only
        threshold = select_threshold(
            val_scores,
            method="percentile",
            percentile=THRESHOLD_PERCENTILE,
        )

        # Final test evaluation
        metrics = evaluate_binary(
            y_test,
            test_scores,
            threshold.threshold,
        )

        predictions = (
            test_scores >= threshold.threshold
        ).astype(int)

        runtime = {
            "train_seconds": train_runtime,
            "score_seconds": score_runtime,
            "total_seconds": time.perf_counter() - start_time,
            "device": DEVICE,
        }

        resolved = {
            "dataset": DATASET,
            "mode": MODE,
            "seed": seed,
            "lookback": LOOKBACK,
            "threshold_method": "percentile",
            "threshold_percentile": THRESHOLD_PERCENTILE,
            "threshold_value": float(threshold.threshold),
            "source_info": source_info,
            "model": model.export_configuration(),
        }

        out = {
            "dataset": DATASET,
            "model": "gdn",
            "seed": seed,
            "precision": metrics["precision"],
            "recall": metrics["recall"],
            "f1": metrics["f1"],
            "auroc": metrics["auroc"],
            "auprc": metrics["auprc"],
            "threshold": float(threshold.threshold),
            "runtime": runtime,
            "evaluation_protocol": (
                "point-wise/window-final-label; "
                "validation-only percentile threshold; "
                "no point adjustment"
            ),
            "warnings": metrics.get("warnings", []),
        }

        # Save model and numerical outputs
        model.save(run_dir / "checkpoint.pt")

        np.savez_compressed(
            run_dir / "scores_predictions.npz",
            test_scores=test_scores,
            predictions=predictions,
            y_test=y_test,
            validation_scores=val_scores,
        )

        # Save logs and configuration
        (run_dir / "training_log.json").write_text(
            json.dumps(model.training_log_, indent=2),
            encoding="utf-8",
        )

        (run_dir / "runtime.json").write_text(
            json.dumps(runtime, indent=2),
            encoding="utf-8",
        )

        (run_dir / "resolved_config.yaml").write_text(
            yaml.safe_dump(
                resolved,
                sort_keys=False,
            ),
            encoding="utf-8",
        )

        torch_version = (
            getattr(torch, "__version__", "unknown")
            if "torch" in globals()
            else "not imported"
        )

        environment_text = (
            f"python={sys.version}\n"
            f"platform={platform.platform()}\n"
            f"torch={torch_version}\n"
            f"device={DEVICE}\n"
        )

        (run_dir / "environment.txt").write_text(
            environment_text,
            encoding="utf-8",
        )

        # Save metrics last so its presence indicates a completed run
        metrics_path.write_text(
            json.dumps(out, indent=2),
            encoding="utf-8",
        )

        pd.DataFrame([out]).to_csv(
            run_dir / "metrics.csv",
            index=False,
        )

        # Clear any failure left by an earlier interrupted run
        failure_path.write_text(
            "",
            encoding="utf-8",
        )

        print(
            f"Completed seed {seed}: "
            f"precision={metrics['precision']:.4f}, "
            f"recall={metrics['recall']:.4f}, "
            f"F1={metrics['f1']:.4f}"
        )

        return out

    except Exception:
        failure_path.write_text(
            traceback.format_exc(),
            encoding="utf-8",
        )

        print(
            f"Seed {seed} failed. "
            f"Traceback saved to {failure_path}"
        )

        raise


# Run all configured seeds
results = [
    run_seed(seed)
    for seed in RUN_CFG["seeds"]
]

# Display summary
summary = pd.DataFrame(results)

summary[
    [
        "dataset",
        "model",
        "seed",
        "precision",
        "recall",
        "f1",
        "auroc",
        "auprc",
        "threshold",
        "evaluation_protocol",
    ]
]

/usr/local/lib/python3.12/dist-packages/numpy/lib/_function_base_impl.py:2922: RuntimeWarning: invalid value encountered in divide
  c /= stddev[:, None]
/usr/local/lib/python3.12/dist-packages/numpy/lib/_function_base_impl.py:2923: RuntimeWarning: invalid value encountered in divide
  c /= stddev[None, :]


Completed seed 42: precision=0.1806, recall=0.9036, F1=0.3010


/usr/local/lib/python3.12/dist-packages/numpy/lib/_function_base_impl.py:2922: RuntimeWarning: invalid value encountered in divide
  c /= stddev[:, None]
/usr/local/lib/python3.12/dist-packages/numpy/lib/_function_base_impl.py:2923: RuntimeWarning: invalid value encountered in divide
  c /= stddev[None, :]


Completed seed 43: precision=0.2597, recall=0.8276, F1=0.3954


/usr/local/lib/python3.12/dist-packages/numpy/lib/_function_base_impl.py:2922: RuntimeWarning: invalid value encountered in divide
  c /= stddev[:, None]
/usr/local/lib/python3.12/dist-packages/numpy/lib/_function_base_impl.py:2923: RuntimeWarning: invalid value encountered in divide
  c /= stddev[None, :]


Completed seed 44: precision=0.1824, recall=0.9024, F1=0.3035


,dataset,model,seed,precision,recall,f1,auroc,auprc,threshold,evaluation_protocol
0,swat,gdn,42,0.180612,0.903579,0.301049,0.876530,0.762580,106.025594,point-wise/window-final-label; validation-only...
1,swat,gdn,43,0.259732,0.827647,0.395385,0.872130,0.763051,143.677409,point-wise/window-final-label; validation-only...
2,swat,gdn,44,0.182426,0.902373,0.303497,0.877276,0.762157,103.051060,point-wise/window-final-label; validation-only...
